# Imports

In [ ]:
import behaviors
import no_signaling_sets
import numpy as np
import samplers

from tqdm import tqdm

In [ ]:
delta = 2
m = 2

sampler = samplers.NoSignalingSampler(delta, m)
srns_set = no_signaling_sets.ShortRangeNoSignalingSet(delta, m)

# Run once

In [ ]:
sampled_behavior = sampler.sample()

result = srns_set.lp_test(sampled_behavior)

print(f"alpha value: {-result.fun}")

## Check the closest SRNS behavior

In [ ]:
yielded_behavior = behaviors.LatentSRNSBehavior(
    delta=delta,
    m=m,
    vector=np.clip(np.array(result.x[1:]), 0, 1),
)

print(f"Yielded behavior is {yielded_behavior}")
print(f"Yielded behavior is tested [{yielded_behavior.is_no_signaling()}] to being no signaling")


## Sanity check

In [ ]:
print(f"PR box is tested [{srns_set.is_in_set(behaviors.pr_box)}] to being SRNS")


# Run on a batch and color the SRNS set

In [ ]:
from time import time

delta = 2
m = 2

sampler = samplers.NoSignalingSampler(delta, m)
srns_set = no_signaling_sets.ShortRangeNoSignalingSet(delta, m)

n_samples = int(2e6)

In [ ]:
sample_arr = sampler.sample_multiple(number_of_samples=n_samples)
files_suffix = f"delta_{delta}_m_{m}_samples_{n_samples}_runtime_{time()}"

In [ ]:
np.save(f"../data/view_srns/sampled_behaviors_{files_suffix}.npy", sample_arr)

In [ ]:
sample_list = list(sample_arr)

In [ ]:
belonging_list = []
for i, vector in enumerate(tqdm(sample_list)):
    behavior = behaviors.RoutedBehavior(
        delta=delta,
        m=m,
        vector=vector,
    )
    belonging_list.append([i, srns_set.is_in_set(behavior)])

np.save(f"../data/view_srns/belonging_list_{files_suffix}.npy", belonging_list)

In [ ]:
analyzer = samplers.SamplesAnalyzer(
    delta=delta,
    m=m,
    samples=sample_arr,
)

In [ ]:
analyzer.plot_projection(
    save_path=f"../data/view_srns/projection_{files_suffix}.png",
    plot=False,
    colors=belonging_list,
    )

In [ ]:
np.mean(np.array(belonging_list)[:, 1])